# IRWA Final Project – Part 2: Indexing and Evaluation

## 0. Setup and paths

In [117]:
import json
import re
import csv
import math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download("stopwords", quiet=True)
STOPWORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()

# Paths relative to project_progress/part_2/ (data/ at the repository root).
# Change them only if your data files are somewhere else.

DATA_PATH = "../../data/fashion_products_dataset.json"
LABELS_PATH = "../../data/validation_labels.csv"

#DATA_PATH = "/Users/mrv/Desktop/data/fashion_products_dataset.json"
#LABELS_PATH = "/Users/mrv/Desktop/data/validation_labels.csv"

## 1. Pre-processing (Part 1)

In [119]:
DOMAIN_STOPWORDS = set()  # empty by decision (see Part 1 report)


def tokenize(text):
    if not text:
        return []
    return re.findall(r"[a-z0-9]+", str(text).lower())


def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOPWORDS and t not in DOMAIN_STOPWORDS]


def stem_tokens(tokens):
    return [STEMMER.stem(t) for t in tokens]


def preprocess_text(text):
    return stem_tokens(remove_stopwords(tokenize(text)))


def product_details_to_text(details):
    if not details:
        return ""
    parts = []
    for item in details:
        if isinstance(item, dict):
            for key, value in item.items():
                parts.append(f"{key} {value}")
        else:
            parts.append(str(item))
    return " ".join(parts)

## 2. Build the inverted index

In [121]:
#DECISION: we index all three text fields in one bag of terms. Each product gets one doc_norm, the Euclidean norm of its term-frequency vector, used to normalise the TF weights. We do not keep field information in the index; the trade-off is discussed in the report.
#DECISION: doc_id is the product pid, because the validation labels refer to products by pid.

with open(DATA_PATH, encoding="utf-8") as f:
    raw_docs = json.load(f)

N = len(raw_docs)
index = defaultdict(dict)   # term -> {pid: raw tf}
doc_norm = {}               # pid -> Euclidean norm of the document's tf vector
display = {}                # pid -> fields to show in results

for raw in raw_docs:
    pid = raw["pid"]
    tokens = (preprocess_text(raw.get("title"))+preprocess_text(raw.get("description"))+preprocess_text(product_details_to_text(raw.get("product_details"))))
    tf = Counter(tokens)
    for term, count in tf.items():
        index[term][pid] = count
    doc_norm[pid] = math.sqrt(sum(c * c for c in tf.values())) or 1.0
    display[pid] = {k: raw.get(k) for k in ["title", "brand", "sub_category", "selling_price", "average_rating", "out_of_stock", "url"]}

print("Documents indexed:", N)
print("Distinct terms in the index:", len(index))

Documents indexed: 28080
Distinct terms in the index: 20426


## 3. Conjunctive (AND) search and TF-IDF ranking

In [123]:
#DECISION: the query norm is computed over the query terms only, which does not change the ranking of the documents for a given query (the norm is the same for all documents), but makes the scores comparable between queries.

def idf(term):
    df_t = len(index.get(term, {}))
    return math.log(N / df_t) if df_t else 0.0


def search_and(query):
    #Return the pids that contain ALL query terms (conjunctive match).
    terms = list(dict.fromkeys(preprocess_text(query)))  # unique terms, in order
    if not terms:
        return [], terms
    postings = [set(index.get(t, {}).keys()) for t in terms]
    if any(len(p) == 0 for p in postings):
        return [], terms  # one term does not occur at all: no document can match
    matches = set.intersection(*postings)
    return list(matches), terms


def rank_tfidf(query):
    #Return a list of (pid, score) for the AND matches, sorted by score (descending).
    matches, terms = search_and(query)
    if not matches:
        return []
    q_tf = Counter(preprocess_text(query))
    q_norm = math.sqrt(sum(c * c for c in q_tf.values())) or 1.0
    q_weights = {t: (c / q_norm) * idf(t) for t, c in q_tf.items() if t in index}
    scores = {}
    for pid in matches:
        s = 0.0
        for t, qw in q_weights.items():
            tf_d = index[t][pid]
            s += qw * (tf_d / doc_norm[pid]) * idf(t)
        scores[pid] = s
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def show_results(query, top=10):
    ranked = rank_tfidf(query)
    print(f"Query: {query!r} -> {len(ranked)} AND matches")
    for pid, score in ranked[:top]:
        d = display[pid]
        print(f"  {score:.4f} | {pid} | {d['title']}")
    print()


show_results("women full sleeve sweatshirt cotton")
show_results("men slim jeans blue")

Query: 'women full sleeve sweatshirt cotton' -> 500 AND matches
  2.5683 | SWSFMTNHEM5SNHVZ | Full Sleeve Solid Women Sweatshirt
  2.2327 | SWSFMJGS8HVBPEH6 | Full Sleeve Solid Women Sweatshirt
  2.2088 | SWSFMJGTVFW3NAXZ | Full Sleeve Solid Women Sweatshirt
  2.2088 | SWSFMJGTQFMZGZQH | Full Sleeve Solid Women Sweatshirt
  2.2088 | SWSFMJGTGZMJXK5H | Full Sleeve Solid Women Sweatshirt
  2.1083 | SWSEP4DJBZWHAZXC | Full Sleeve Self Design Women Sweatshirt
  2.0989 | SWSEP4EBPJYA6G2S | Full Sleeve Self Design Women Sweatshirt
  2.0870 | SWSFXZWGMGVDUCXA | Full Sleeve Printed Women Sweatshirt
  2.0573 | SWSFYFFYFXCMPBDZ | Full Sleeve Printed Women Sweatshirt
  2.0573 | SWSFYFFYQ7Z3ZKN6 | Full Sleeve Printed Women Sweatshirt

Query: 'men slim jeans blue' -> 222 AND matches
  1.8917 | JEAFDZRVABZZ2PQ3 | Stretchable Slim Men Blue Jeans  (Pack of 2)
  1.8651 | JEAFRAQXEKGUPNUN | Slim Men Blue Jeans
  1.8643 | JEAFSKYHTE76YWH7 | Slim Men Blue Jeans
  1.8643 | JEAFSKYHRVZSABPR | Slim Men Blue 

## 4. Five test queries

In [125]:
#DECISION: queries use terms that occur frequently in the catalogue, so that the AND semantics still return results.

TEST_QUERIES = [
    "men polo shirt",
    "women printed kurta",
    "blue slim jeans",
    "women sweatshirt cotton",
    "men track pants",
]

for q in TEST_QUERIES:
    matches, terms = search_and(q)
    print(f"{q!r:32s} terms={terms}  AND matches={len(matches)}")

'men polo shirt'                 terms=['men', 'polo', 'shirt']  AND matches=1466
'women printed kurta'            terms=['women', 'print', 'kurta']  AND matches=42
'blue slim jeans'                terms=['blue', 'slim', 'jean']  AND matches=442
'women sweatshirt cotton'        terms=['women', 'sweatshirt', 'cotton']  AND matches=513
'men track pants'                terms=['men', 'track', 'pant']  AND matches=877


### 4.1 Candidate lists for relevance judgement

In [127]:
candidates = []
for q in TEST_QUERIES:
    ranked = rank_tfidf(q)
    print(f"=== {q} ({len(ranked)} matches; showing top 20) ===")
    for pid, score in ranked[:20]:
        d = display[pid]
        print(f"  {pid} | {score:.4f} | {d['sub_category']} | {d['selling_price']} | {d['title']}")
        candidates.append({"query": q, "pid": pid, "title": d["title"], "score": round(score, 4)})
    print()

import os
# Save next to the dataset
CANDIDATES_PATH = os.path.join(os.path.dirname(DATA_PATH), "test_query_candidates.csv")
pd.DataFrame(candidates).to_csv(CANDIDATES_PATH, index=False)
print("Candidate list saved to", CANDIDATES_PATH)

=== men polo shirt (1466 matches; showing top 20) ===
  TSHFWCPHJXZHYPNP | 1.5578 | Topwear | 499 | Solid Men Polo Neck Beige T-Shirt
  TSHFWCPH77RAYAFF | 1.5578 | Topwear | 499 | Solid Men Polo Neck White T-Shirt
  TSHFWCPH3K9Q9PGX | 1.5578 | Topwear | 499 | Solid Men Polo Neck Purple T-Shirt
  TSHFWCPH58HTE4FT | 1.5578 | Topwear | 499 | Solid Men Polo Neck White T-Shirt
  TSHFWCPHVBZRCCYQ | 1.5554 | Topwear | 499 | Solid Men Polo Neck Blue T-Shirt
  TSHFWCPHG3ZSARY3 | 1.5554 | Topwear | 499 | Solid Men Polo Neck Green T-Shirt
  TSHFWCPHA8JF4HDV | 1.5554 | Topwear | 499 | Solid Men Polo Neck Blue T-Shirt
  TSHFWCPHK7J5H2UU | 1.5554 | Topwear | 499 | Solid Men Polo Neck Green T-Shirt
  TSHFWCPHZEAZXY9K | 1.5530 | Topwear | 499 | Solid Men Polo Neck Dark Blue T-Shirt
  TSHFWCPHW85DNJFF | 1.5482 | Topwear | 499 | Solid Men Polo Neck Red T-Shirt
  TSHFWJZEBEGPHCTH | 1.5217 | Topwear | 499 | Solid Men Polo Neck Grey T-Shirt
  TSHFWJZEJMHYGR3D | 1.5217 | Topwear | 499 | Solid Men Polo Neck 

## 5. Evaluation metrics

In [129]:
#DECISION: P@K divides by K even when fewer than K documents are retrieved, because a short result list should not get full precision.
#DECISION: AP@K divides by the number of relevant documents within the top K, as in the lab notebook.


def precision_at_k(ranked, k):
    return sum(ranked[:k]) / k


def recall_at_k(ranked, total_relevant, k):
    if total_relevant == 0:
        return 0.0
    return sum(ranked[:k]) / total_relevant


def f1_at_k(ranked, total_relevant, k):
    p = precision_at_k(ranked, k)
    r = recall_at_k(ranked, total_relevant, k)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def average_precision_at_k(ranked, k):
    hits, precisions = 0, []
    for i, rel in enumerate(ranked[:k], start=1):
        if rel == 1:
            hits += 1
            precisions.append(hits / i)
    return sum(precisions) / hits if hits else 0.0


def reciprocal_rank(ranked, k):
    for i, rel in enumerate(ranked[:k], start=1):
        if rel == 1:
            return 1 / i
    return 0.0


def dcg_at_k(gains, k):
    gains = np.asarray(gains, dtype=float)[:k]
    if gains.size == 0:
        return 0.0
    discounts = np.log2(np.arange(gains.size) + 2)
    return float(np.sum((2 ** gains - 1) / discounts))


def ndcg_at_k(ranked, ideal_gains, k):
    ideal = dcg_at_k(sorted(ideal_gains, reverse=True), k)
    return 0.0 if ideal == 0 else dcg_at_k(ranked, k) / ideal

## 6. Evaluation on the validation queries

In [131]:
#DECISION: the validation file has only binary labels, so the NDCG gains are the same binary values (1 or 0).

VALIDATION_QUERIES = {
    "women full sleeve sweatshirt cotton": "Query 1",
    "men slim jeans blue": "Query 2",
}

labels_df = pd.read_csv(LABELS_PATH)
print("Labelled pairs:", len(labels_df), "| queries:", labels_df["query_id"].nunique())

K = 10
per_query = {}
for query, name in VALIDATION_QUERIES.items():
    query_id = 1 if name == "Query 1" else 2
    labelled = labels_df[labels_df["query_id"] == query_id]
    rel_map = dict(zip(labelled["pid"], labelled["labels"]))
    total_relevant = int(sum(rel_map.values()))

    ranked_pids = [pid for pid, _ in rank_tfidf(query) if pid in rel_map]   # labelled, retrieved, ranked
    ranked_labels = [int(rel_map[pid]) for pid in ranked_pids]
    ideal_gains = list(rel_map.values())

    per_query[name] = {
        "P@10": precision_at_k(ranked_labels, K),
        "R@10": recall_at_k(ranked_labels, total_relevant, K),
        "AP@10": average_precision_at_k(ranked_labels, K),
        "F1@10": f1_at_k(ranked_labels, total_relevant, K),
        "RR@10": reciprocal_rank(ranked_labels, K),
        "NDCG@10": ndcg_at_k(ranked_labels, ideal_gains, K),
    }

MAP_10 = float(np.mean([per_query[n]["AP@10"] for n in per_query]))
MRR_10 = float(np.mean([per_query[n]["RR@10"] for n in per_query]))

for name, m in per_query.items():
    print(name)
    print(f"  P@10={m['P@10']:.3f}  R@10={m['R@10']:.3f}  AP@10={m['AP@10']:.3f}  "
          f"F1@10={m['F1@10']:.3f}  NDCG@10={m['NDCG@10']:.3f}")
print(f"MAP@10={MAP_10:.3f}")
print(f"MRR@10={MRR_10:.3f}")

Labelled pairs: 40 | queries: 2
Query 1
  P@10=1.000  R@10=0.769  AP@10=1.000  F1@10=0.870  NDCG@10=1.000
Query 2
  P@10=0.600  R@10=0.600  AP@10=0.915  F1@10=0.600  NDCG@10=0.706
MAP@10=0.958
MRR@10=1.000


### 6.1 Detail: retrieved labelled products for each query

In [133]:
for query, name in VALIDATION_QUERIES.items():
    query_id = 1 if name == "Query 1" else 2
    labelled = labels_df[labels_df["query_id"] == query_id]
    rel_map = dict(zip(labelled["pid"], labelled["labels"]))
    print(f"=== {name}: {query!r} ===")
    print(f"  labelled relevant: {sum(rel_map.values())} of {len(rel_map)}")
    retrieved = {pid for pid, _ in rank_tfidf(query)}
    for pid, label in rel_map.items():
        status = "retrieved" if pid in retrieved else "NOT retrieved"
        print(f"  label={label} | {status:13s} | {display[pid]['title']}")
    print()

=== Query 1: 'women full sleeve sweatshirt cotton' ===
  labelled relevant: 13 of 20
  label=1 | retrieved     | Full Sleeve Printed Women Sweatshirt
  label=0 | retrieved     | Full Sleeve Striped Women Sweatshirt
  label=1 | retrieved     | Full Sleeve Printed Women Sweatshirt
  label=1 | retrieved     | Full Sleeve Graphic Print Women Sweatshirt
  label=0 | retrieved     | Full Sleeve Solid Women Sweatshirt
  label=1 | retrieved     | Full Sleeve Graphic Print Women Sweatshirt
  label=0 | retrieved     | Typography Women V Neck Multicolor T-Shirt  (Pack of 2)
  label=1 | retrieved     | Full Sleeve Printed Women Sweatshirt
  label=1 | retrieved     | Full Sleeve Solid Women Sweatshirt
  label=1 | retrieved     | Full Sleeve Graphic Print Women Sweatshirt
  label=1 | retrieved     | Full Sleeve Graphic Print Women Sweatshirt
  label=0 | retrieved     | Typography Women V Neck Multicolor T-Shirt  (Pack of 2)
  label=0 | retrieved     | Full Sleeve Printed Women Sweatshirt
  label=0 | 